# Does a 7B judge see what the outcome sees?

Measuring a hint judge before any policy trains against it. Two judges have
already failed this on measurement rather than on inspection:

| | coverage | variance | LOST vs WON |
|---|---|---|---|
| technique vocabulary | 10% | 0.10 | n/a, too sparse |
| 1.5B judge, first prompt | 100% | **0.0000** | constant, MISLEADING everywhere |
| 1.5B judge, reframed | 85% | 0.607 | **+0.545 vs +0.500** -- cannot tell them apart |

Coverage and variance are solved. The open question is the third column: does
the judge score the dialogues where tutoring *rescued* a problem above the ones
where tutoring *destroyed* it? A judge that supplies gradient without that is a
target to be optimised, not a measurement, which is the v16 failure wearing a
better hat.

The 1.5B judge could not. A 3B could not be tested on a 16 GB laptop -- it hit
15 GB of swap. This runs 7B on a T4.

**4-bit is required, not an optimisation.** Qwen2.5-7B is ~15.2 GB in bfloat16
against the T4's 15.6 GB, which leaves no room for activations; 4-bit is
~4.6 GB. bitsandbytes is CUDA-only, which is exactly why this cannot run on
Apple Silicon.

## 1. Environment

In [ ]:
# Kaggle preinstalls torch/transformers/peft/bitsandbytes/datasets. Uncomment
# only if an import below actually fails.
# !pip install -q transformers peft accelerate bitsandbytes pydantic typer datasets

# Kaggle's image ships torchao 0.10.0, but the installed peft requires >= 0.16.0
# and its is_torchao_available() *raises* ImportError instead of returning False.
# That kills get_peft_model() even though this project never uses torchao
# quantization. Removing the package makes peft skip the torchao dispatcher
# cleanly and fall through to the standard LoRA path.
import importlib.metadata
import importlib.util
import subprocess
import sys


def drop_incompatible_torchao(minimum=(0, 16, 0)):
    if importlib.util.find_spec("torchao") is None:
        print("torchao not installed — nothing to do.")
        return
    try:
        raw = importlib.metadata.version("torchao")
        parsed = tuple(int(p) for p in raw.split(".")[:3])
    except Exception:
        raw, parsed = "unknown", (0, 0, 0)

    if parsed >= minimum:
        print(f"torchao {raw} is compatible with peft — leaving it installed.")
        return

    print(f"Removing torchao {raw} (peft needs >= {'.'.join(map(str, minimum))})...")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
        check=False,
    )
    importlib.invalidate_caches()
    still_there = importlib.util.find_spec("torchao") is not None
    print("torchao removed." if not still_there else "WARNING: torchao still importable.")


drop_incompatible_torchao()

def ensure_bitsandbytes(minimum="0.46.1"):
    """`Settings.kaggle()` sets `student_quantize_4bit=True`, which needs this.

    Historically preinstalled on Kaggle's image; stopped being true at some
    point and broke a run at the very first model load with a plain
    ImportError. Checking and installing defensively means a future image
    change fails the same way silently instead of wasting a GPU session.
    """
    if importlib.util.find_spec("bitsandbytes") is not None:
        print("bitsandbytes already installed — nothing to do.")
        return
    print(f"bitsandbytes missing — installing >= {minimum}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"bitsandbytes>={minimum}"],
        check=True,
    )
    importlib.invalidate_caches()
    ok = importlib.util.find_spec("bitsandbytes") is not None
    print("bitsandbytes installed." if ok else "WARNING: bitsandbytes still missing.")


ensure_bitsandbytes()


def ensure_datasets(minimum="2.14.0"):
    """`load_mbpp` needs this; docs already flagged it as unreliably present
    alongside bitsandbytes, so check it the same defensive way."""
    if importlib.util.find_spec("datasets") is not None:
        print("datasets already installed — nothing to do.")
        return
    print(f"datasets missing — installing >= {minimum}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"datasets>={minimum}"],
        check=True,
    )
    importlib.invalidate_caches()
    ok = importlib.util.find_spec("datasets") is not None
    print("datasets installed." if ok else "WARNING: datasets still missing.")


ensure_datasets()


## 2. Locate the sahai package

In [ ]:
import os
import sys
import zipfile


def find_sahai_root(base="/kaggle/input"):
    """Locate the directory containing the sahai package.

    The mount path is not predictable: a dataset attached via
    kernel-metadata.json lands under /kaggle/input/datasets/<owner>/<slug>/,
    while one attached in the browser lands at /kaggle/input/<slug>/. Walk
    instead of guessing, and fall back to extracting sahai.zip if Kaggle
    served the archive without expanding it.
    """
    if not os.path.isdir(base):
        return None  # running locally

    for root, dirs, files in os.walk(base):
        if os.path.basename(root) == "sahai" and "__init__.py" in files:
            return os.path.dirname(root)

    for root, dirs, files in os.walk(base):
        if "sahai.zip" in files:
            dest = "/kaggle/working/_sahai_src"
            os.makedirs(dest, exist_ok=True)
            with zipfile.ZipFile(os.path.join(root, "sahai.zip")) as zf:
                zf.extractall(dest)
            print(f"Extracted sahai.zip -> {dest}")
            # The archive may or may not contain a top-level sahai/ directory.
            if os.path.isdir(os.path.join(dest, "sahai")):
                return dest
            return os.path.dirname(dest)

    return None


sahai_root = find_sahai_root()
if sahai_root:
    sys.path.insert(0, sahai_root)
    print(f"Found sahai package in: {sahai_root}")
elif os.path.isdir("/kaggle/input"):
    print("ERROR: sahai package not found. Full /kaggle/input tree:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count(os.sep) - 2
        print(f"{'  ' * depth}{os.path.basename(root) or root}/")
        for f in sorted(files)[:10]:
            print(f"{'  ' * (depth + 1)}{f}")

import sahai
print(f"SAHAI version: {sahai.__version__}")

## 3. GPU check

In [ ]:
import torch
from sahai.core.models import load_for_training, load_for_inference

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute capability: sm_{capability[0]}{capability[1]}")

    # Kaggle's PyTorch build has no sm_60 kernels, and bitsandbytes 4-bit needs
    # Turing or newer, so a P100 cannot run this stack at all. Stop immediately
    # rather than failing later inside model loading with an opaque
    # "no kernel image is available for execution on the device".
    #
    # Note: `--accelerator gpu-t4x2` is accepted by the CLI but ignored by the
    # server, so the accelerator must be set in the Kaggle notebook sidebar.
    if capability < (7, 0):
        raise RuntimeError(
            f"This GPU (sm_{capability[0]}{capability[1]}) cannot run this stack: "
            f"PyTorch here requires sm_70+ and bitsandbytes 4-bit requires sm_75+. "
            f"Open the notebook on kaggle.com and set Accelerator = 'GPU T4 x2' "
            f"in the right-hand sidebar, then Save & Run All."
        )
else:
    raise RuntimeError("No GPU. Set enable_gpu in kernel-metadata.json.")

## 4. Find the saved dialogues

180 transcripts from the 2026-10-02 M4 run, shipped in the dataset. Judging
saved dialogues rather than generating new ones keeps this cheap and holds the
transcripts fixed, so the only thing varying between judge experiments is the
judge.

In [ ]:
import glob, json, os

cands = glob.glob("/kaggle/input/**/ab_benchmark.json", recursive=True)
assert cands, "ab_benchmark.json not found under /kaggle/input"
DIALOGUES = cands[0]
d = json.load(open(DIALOGUES))
print("dialogues:", DIALOGUES)
print("  source :", d.get("source"))
print("  arms   :", {k: len(v["dialogues"]) for k, v in d["arms"].items()})
print("  tutor turns to judge:",
      sum(1 for a in ("base", "trained") for x in d["arms"][a]["dialogues"]
          for t in x["turns"] if t["role"] == "tutor"))

## 5. Run the measurement

~200 tutor turns per arm, one short greedy generation each. The prompt carries
the problem and its reference solution, so most of the cost is prefill.

In [ ]:
import sys, time

sys.argv = ["judge_validation",
            "--judge", "Qwen/Qwen2.5-7B-Instruct",
            "--dialogues", DIALOGUES,
            "--arms", "base,trained",
            "--quantize-4bit",
            "--out", "/kaggle/working/judge_validation_7b.json"]

t0 = time.time()
from sahai.eval.judge_validation import main
main()
print(f"\ntotal {(time.time()-t0)/60:.1f} min")

## 6. Verdict

In [ ]:
import json

r = json.load(open("/kaggle/working/judge_validation_7b.json"))
print(f"judge: {r['judge']}  (4-bit: {r['quantize_4bit']})\n")
print(f"{'arm':<9} {'coverage':>9} {'variance':>9} {'corr':>8}")
print("-" * 38)
for arm, a in r["arms"].items():
    print(f"{arm:<9} {a['coverage']:>8.0%} {a['variance']:>9.4f} {a['corr_with_solved']:>+8.3f}")

lost, won = r.get("lost_mean"), r.get("won_mean")
print("\nTHE DECIDING TEST")
if lost is not None and won is not None:
    print(f"  tutoring LOST the problem : judge {lost:+.3f}")
    print(f"  tutoring WON  the problem : judge {won:+.3f}")
    print(f"  gap (WON - LOST)          : {won-lost:+.3f}")
    print()
    if won - lost > 0.15:
        print("  The judge ranks helpful tutoring above harmful tutoring.")
        print("  Worth training against -- set mu_correct > 0 and re-measure after.")
    elif won - lost > 0.0:
        print("  Ordered correctly but weakly. Larger than the 1.5B judge's")
        print("  -0.045, still thin enough that training would mostly optimise")
        print("  the judge rather than the teaching.")
    else:
        print("  The judge does NOT see what the outcome sees, same as 1.5B.")
        print("  Scale is not the missing ingredient. Training against this")
        print("  would optimise the judge, which is the v16 failure again.")
else:
    print("  unaided arm absent, cannot compute")